## Notebook 1: Preperation

#### This notebook is the first of two notebooks pertaining the empirical analysis of this thesis. Notebook 1 includes the data preperation process. Notebook 2 includes the training of the models and the application of the in- and post-processors.

## Bachelor´s Thesis 
#### for acquiring the degree of Bachelor of Science (B.Sc.) at the School of Business and Economics of Humboldt-Universität zu Berlin

### Topic: Algorithmic Fairness at the Intersection of Gender and Age - Evaluating In- and Post-Processing Methods on Credit Risk Data


#### Name: Celina Gina Kaliman
#### Student number: 625187
#### E-Mail address: celina.gina.kaliman@student.hu-berlin.de
#### First Examiner: Prof. Dr. Lessmann
#### Second Examiner: Prof. Dr. Mendling

##### In order to recreate the same running environment, run this command in your terminal: pip install -r requirements.txt

In [1]:
#installing the necessary packages
%pip install fairlearn xgboost scikit-learn pandas numpy openpyxl ucimlrepo jupyter matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
#importing the necessary libraries 
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from fairlearn.metrics import MetricFrame, demographic_parity_difference, equalized_odds_difference
from sklearn.metrics import precision_score
from fairlearn.reductions import ExponentiatedGradient, GridSearch, DemographicParity, EqualizedOdds
from sklearn.base import BaseEstimator, ClassifierMixin
from fairlearn.postprocessing import ThresholdOptimizer

In [3]:
#set random seed for reproducibility
random_seed = 222

In [4]:
#check the repository for the folder "data" and download the dataset "credit_data.csv" inorder to download the dataset locally. If the dataset is not downloaded locally, it will be taken from the UCI repository on the open web via the following code.
if os.path.exists("data/credit_data.csv"):
    df = pd.read_csv("data/credit_data.csv")
    print("The dataset was downloaded locally.")
else:
    data = fetch_ucirepo(id=350)
    X = data.data.features
    y = data.data.targets
    df = X.copy()
    df["default"] = y
    os.makedirs("data", exist_ok=True)
    df.to_csv("data/credit_data.csv", index=False)
    print("The dataset was taken from the UCI repository.")

The dataset was downloaded locally.


In [5]:
#check the shape of the dataset and it´s original feature and column names
print(df.shape)
df.head()

(30000, 24)


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [6]:
#remove duplicates and missing values
df = df.drop_duplicates()
print(df.shape)
df = df.dropna()
print(df.shape)

(29965, 24)
(29965, 24)


In [7]:
#relabeling of variables original names to their desprictive names 
rename_map = {
    "X1": "LIMIT_BAL", "X2": "SEX", "X3": "EDUCATION", "X4": "MARRIAGE", "X5": "AGE",
    "X6": "PAY_0", "X7": "PAY_2", "X8": "PAY_3", "X9": "PAY_4", "X10": "PAY_5", "X11": "PAY_6",
    "X12": "BILL_AMT1", "X13": "BILL_AMT2", "X14": "BILL_AMT3", "X15": "BILL_AMT4", "X16": "BILL_AMT5", "X17": "BILL_AMT6",
    "X18": "PAY_AMT1", "X19": "PAY_AMT2", "X20": "PAY_AMT3", "X21": "PAY_AMT4", "X22": "PAY_AMT5", "X23": "PAY_AMT6",
}
df = df.rename(columns=rename_map)

print(df.shape)
df.head()

(29965, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [8]:
#Creating the intersectional groupings
df["gender"] = df["SEX"].map({1: "male", 2: "female"})
df["age_group"] = df["AGE"].apply(lambda a: "under25" if a < 25 else "25plus")
df["intersection"] = df["gender"] + "_" + df["age_group"]


#Identifying class imbalance problem which will be addressed and balanced in the modeling phase in Notebook 2
print("Target balance:")
print(df["default"].value_counts(normalize=True).round(3))

#Displaying the counts of intersectional groups 
print("\nIntersectional subgroup sizes:")
counts = df["intersection"].value_counts()
print(counts)
print("\nSingle attribute group sizes:")
counts = df["gender"].value_counts()
print(counts)
counts = df["age_group"].value_counts()
print(counts)

Target balance:
default
0    0.779
1    0.221
Name: proportion, dtype: float64

Intersectional subgroup sizes:
intersection
female_25plus     16182
male_25plus       11100
female_under25     1909
male_under25        774
Name: count, dtype: int64

Single attribute group sizes:
gender
female    18091
male      11874
Name: count, dtype: int64
age_group
25plus     27282
under25     2683
Name: count, dtype: int64


In [9]:
#building the data frames for the modeling phase. Senstive attributes are excluded from the feature set.
feature_columns = [c for c in df.columns if c not in ["default", "gender", "age_group", "intersection", "SEX", "AGE", "EDUCATION", "MARRIAGE"]]

X = df[feature_columns]
y = df["default"]
sensitive_intersection_df = df["intersection"]
gender_df = df["gender"]
age_df = df["age_group"]

#Test size 30%, train size 70%
X_train, X_test, y_train, y_test, sens_train, sens_test, gender_train, gender_test, age_train, age_test = train_test_split(
    X, y, sensitive_intersection_df, gender_df, age_df, test_size=0.3, random_state=random_seed, stratify=y
)

#Reset variables for better compatibility with fairlearn
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
sens_train = sens_train.reset_index(drop=True)
sens_test = sens_test.reset_index(drop=True)
gender_train = gender_train.reset_index(drop=True)
gender_test = gender_test.reset_index(drop=True)
age_train = age_train.reset_index(drop=True)
age_test = age_test.reset_index(drop=True)

#Check the size of the test set
print("Sizes of test-set of intersectional subgroup:")
test_counts = sens_test.value_counts()
print(test_counts)
print("\nSingle attribute group sizes of test-set:")
print(gender_test.value_counts())
print(age_test.value_counts())

Sizes of test-set of intersectional subgroup:
intersection
female_25plus     4810
male_25plus       3378
female_under25     576
male_under25       226
Name: count, dtype: int64

Single attribute group sizes of test-set:
gender
female    5386
male      3604
Name: count, dtype: int64
age_group
25plus     8188
under25     802
Name: count, dtype: int64


In [10]:
#Scaling of features using StandardScaler for the logistic regression model 
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Scaled feature means:")
print(X_train_scaled.mean().round(3).head())
print("\nScaled feature std devs:")
print(X_train_scaled.std().round(3).head())

Scaled feature means:
LIMIT_BAL    0.0
PAY_0        0.0
PAY_2        0.0
PAY_3        0.0
PAY_4        0.0
dtype: float64

Scaled feature std devs:
LIMIT_BAL    1.0
PAY_0        1.0
PAY_2        1.0
PAY_3        1.0
PAY_4        1.0
dtype: float64


In [11]:
#saving of the prepared data to the folder "data/Output_Notebook1" 
os.makedirs("data/Output_Notebook1", exist_ok=True)
output_notebook1 = "data/Output_Notebook1"
os.makedirs(output_notebook1, exist_ok=True)

X_train.to_csv(f"{output_notebook1}/X_train.csv", index=False)
X_test.to_csv(f"{output_notebook1}/X_test.csv", index=False)
X_train_scaled.to_csv(f"{output_notebook1}/X_train_scaled.csv", index=False)
X_test_scaled.to_csv(f"{output_notebook1}/X_test_scaled.csv", index=False)

y_train.to_csv(f"{output_notebook1}/y_train.csv", index=False)
y_test.to_csv(f"{output_notebook1}/y_test.csv", index=False)

sens_train.to_csv(f"{output_notebook1}/sens_train.csv", index=False, header=["intersection"])
sens_test.to_csv(f"{output_notebook1}/sens_test.csv", index=False, header=["intersection"])
gender_train.to_csv(f"{output_notebook1}/gender_train.csv", index=False, header=["gender"])
gender_test.to_csv(f"{output_notebook1}/gender_test.csv", index=False, header=["gender"])
age_train.to_csv(f"{output_notebook1}/age_train.csv", index=False, header=["age_group"])
age_test.to_csv(f"{output_notebook1}/age_test.csv", index=False, header=["age_group"])

with open(f"{output_notebook1}/config.txt", "w") as f:
    f.write(f"random_seed={random_seed}\n")


print(f"Saved all prepared data to {output_notebook1}/")
print("config.txt was created.")
print("Files:", sorted(os.listdir(output_notebook1)))
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

Saved all prepared data to data/Output_Notebook1/
config.txt was created.
Files: ['.DS_Store', 'X_test.csv', 'X_test_scaled.csv', 'X_train.csv', 'X_train_scaled.csv', 'age_test.csv', 'age_train.csv', 'config.txt', 'gender_test.csv', 'gender_train.csv', 'sens_test.csv', 'sens_train.csv', 'y_test.csv', 'y_train.csv']
X_train shape: (20975, 19), X_test shape: (8990, 19)
